In [2]:
import pandas as pd

### Задание 1

Импортируйте файлы workouts, payments и users. Преобразуйте типы данных "дата+время"

In [8]:
import pandas as pd

workouts = pd.read_csv('workouts.csv')
payments = pd.read_excel('payments.xlsx')
users = pd.read_excel('users.xlsx')

workouts['start_at'] = pd.to_datetime(workouts['start_at'])
payments['payment_date'] = pd.to_datetime(payments['payment_date'])

print(workouts.dtypes)
print(payments.dtypes)
print(users.dtypes)

workout_id                        int64
cost                            float64
start_at                 datetime64[ns]
status                           object
workout_schedule_type            object
client_id                         int64
client_status                    object
workout_type                     object
trainer_department               object
trainer_id                        int64
dtype: object
user_id                  int64
payment_id               int64
workout_type            object
amount                 float64
payment_date    datetime64[ns]
dtype: object
first_contact_datetime     object
age                       float64
workouts_successful         int64
workouts_total              int64
months_active               int64
user_id                     int64
region                     object
geo_group                  object
dtype: object


### Задание 2

~~~ sql
select    workout_type
        , count(workout_id) as cnt
from workouts
group by workout_type
order by cnt desc
~~~

In [9]:
result_2 = workouts.groupby('workout_type').agg(cnt=('workout_id', 'count')).reset_index().sort_values('cnt', ascending=False)
print(result_2)

      workout_type     cnt
2          general  103492
0          cycling    1175
1  functional test      26


### Задание 3

~~~ sql
select    date_part('month', start_at) as dm
        , trainer_department
        , count(workout_id) as cnt
from workouts
group by   dm
         , trainer_department
order by dm
~~~

In [10]:
result_3 = workouts.copy()
result_3['dm'] = result_3['start_at'].dt.month

result_3 = result_3.groupby(['dm', 'trainer_department']).agg(cnt=('workout_id', 'count')).reset_index().sort_values('dm')
print(result_3)

    dm trainer_department   cnt
0    1           Athletic    11
1    1         Consultant   164
2    1            Cycling     1
3    1             Dinamo    36
4    1              Sales   457
..  ..                ...   ...
79  12            Cycling    77
80  12             Dinamo   254
81  12              Sales    22
82  12          Spartacus  2147
83  12    Sports Reserves   192

[84 rows x 3 columns]


### Задание 4

~~~ sql
select    date_trunc('month', payment_date) as p_tm
        , sum(amount)/1000 as sum_amt
from payments
where workout_type = 'general'
group by p_tm
order by p_tm
~~~

In [11]:
result_4 = payments[payments['workout_type'] == 'general'].copy()
result_4['p_tm'] = result_4['payment_date'].dt.to_period('M').dt.to_timestamp()

result_4 = result_4.groupby('p_tm').agg(sum_amt=('amount', 'sum')).reset_index()
result_4['sum_amt'] = result_4['sum_amt'] / 1000
result_4 = result_4.sort_values('p_tm')
print(result_4)

         p_tm       sum_amt
0  2020-01-01   4370.772775
1  2020-02-01   6114.601296
2  2020-03-01   7949.117944
3  2020-04-01  11908.988560
4  2020-05-01  11125.842432
5  2020-06-01   9682.458175
6  2020-07-01   9553.436788
7  2020-08-01   8345.359401
8  2020-09-01   9876.673936
9  2020-10-01  11044.861465
10 2020-11-01  10859.320076
11 2020-12-01   3028.005690


### Задание 5

~~~ sql
select    user_id
        , min(date_trunc('day', payment_date)) as first_date
from payments p
    join users u
        on p.user_id = u.user_id
where region is not null
group by user_id
order by user_id
limit 20
~~~

In [12]:
merged = payments.merge(users, on='user_id', how='inner')

filtered = merged[merged['region'].notnull()].copy()
filtered['first_date'] = filtered['payment_date'].dt.to_period('D').dt.to_timestamp()

result_5 = filtered.groupby('user_id').agg(first_date=('first_date', 'min')).reset_index().sort_values('user_id').head(20)
print(result_5)

    user_id first_date
0       185 2020-04-02
1      1455 2020-02-03
2      1558 2020-05-12
3      5312 2020-02-21
4      5804 2020-01-15
5      6877 2020-11-03
6      7226 2020-03-29
7      7824 2020-02-22
8      8881 2020-04-05
9      8913 2020-08-01
10     8997 2020-06-16
11     9361 2020-05-08
12     9450 2020-02-06
13    10214 2020-11-16
14    11332 2020-02-11
15    15308 2020-04-18
16    15807 2020-11-19
17    16249 2020-01-24
18    17005 2020-10-20
19    17364 2020-10-12


### Задание 6

~~~ sql
select    avg(age) as avg_age
from users p
    left join payments u
        on p.user_id = u.user_id
where payment_date is null
~~~

In [13]:
merged_6 = users.merge(payments, on='user_id', how='left')

no_payment_users = merged_6[merged_6['payment_date'].isnull()]
result_6 = no_payment_users['age'].mean()
print(result_6)

29.5


### Задание 7

~~~ sql
select   p_tm
       , sum(case when workout_type<>'general' then 1.0 else 0.0 end) / count(payment_id) as share_not_general
from payments
group by p_tm
~~~

In [14]:
payments_7 = payments.copy()
payments_7['p_tm'] = payments_7['payment_date'].dt.to_period('M').dt.to_timestamp()
payments_7['not_general'] = (payments_7['workout_type'] != 'general').astype(float)

result_7 = payments_7.groupby('p_tm').agg(
    sum_not_general=('not_general', 'sum'),
    cnt_payment=('payment_id', 'count')
).reset_index()
result_7['share_not_general'] = result_7['sum_not_general'] / result_7['cnt_payment']
result_7 = result_7[['p_tm', 'share_not_general']]
print(result_7)

         p_tm  share_not_general
0  2020-01-01           0.004255
1  2020-02-01           0.012085
2  2020-03-01           0.006969
3  2020-04-01           0.007009
4  2020-05-01           0.003317
5  2020-06-01           0.010436
6  2020-07-01           0.007656
7  2020-08-01           0.007642
8  2020-09-01           0.017479
9  2020-10-01           0.017843
10 2020-11-01           0.019950
11 2020-12-01           0.020528


### Задание 8

~~~ sql
select    w.clientId
        , cnt_workout
        , amt
from (select    client_id
              , count(workout_id) as cnt_workout
     from workouts
     group by client_id
     ) w
    join (select      user_id
                    , sum(amount) as amt
          from payments
          group by user_id
          ) p
        on p.user_id = w.client_id
limit 10
~~~

In [15]:
workouts_agg = workouts.groupby('client_id').agg(cnt_workout=('workout_id', 'count')).reset_index()
payments_agg = payments.groupby('user_id').agg(amt=('amount', 'sum')).reset_index()

result_8 = workouts_agg.merge(payments_agg, left_on='client_id', right_on='user_id', how='inner')
result_8 = result_8[['client_id', 'cnt_workout', 'amt']].head(10)
print(result_8)

   client_id  cnt_workout            amt
0        185           50   29033.865003
1       1455           64  149649.891518
2       1558           13   18259.280262
3       5312           16   28416.673028
4       5804           46   38121.522478
5       6877           10   18460.888941
6       7226           18   27234.575055
7       7824            9    9177.146485
8       8881           52   28763.182307
9       8913           38   47770.522298
